# Deep Q-Networks (2-20-26)

## Introduction
* Reinforcement learning algorithm that combines Q-learning with neural networks to solve problems in complex spaces
* Fun fact: Introduced by DeepMind and achieved human-level perforamnce on Atari video games

## Break Down of Q-Learning
* Q-Learning is based on the Q-table (Q(s,a)), which stores one Q-value per possible state and action. 
    * The Q-values can be interpreted as the total possible reward if action a is taken at state s and all behavior after the action is taken is optimal.
* This works well when there is a small number of states and actions (and thus the Q-table is small) and there is only one value to be stored per state, action pair.
* Conversely, Q-Learning breaks down, or becomes inefficient, when the number of possible states and/or actions is large or more than one value needs to be stored per state, action pair.
    * Q-tables do not scale well

## Difference from Q-Learning
* The core idea of a Deep Q-Network is that the Q-Table, Q(s,a) is replaced with a neural network $Q_\theta(s,a)$.


## Deep Q-Learning Overview
* **Input:** A representation of the state, typically one-hot encoded. 
* **Output:** A vector of Q-values, one per action that could be taken.
    * Example: For GridWorld there would be four outputs, one q-value per up, down, left, and right.
* $\epsilon$-greedy strategy is still used through the agent typically takes the action resulting in the highest Q-value

## Main Components

### Experience Relay
* Also called replay buffer or replay memory
* Stpres past interactions between the agent and the environment:
    * Stores the current state, the action, the reward recieved for the action, the new state, and if the goal was reached.
* When being trained the neural network will randomly sample from the experience relay to learn from past experiences in addition to the current experience
* The experience relay is the work around for neural networks assuming data points are independent but reinforcement learning generating data sequentially. Additionally, without the experience buffer the experience is learning from once and then is discarded. 

### Target Network
* The the target network is a second neural network, with the same architecture as the one which is deciding the Q-values, but that learns more slowly.
* It provides a stable target value which is used in training the main neural network. 
* Without the target network the training can oscilate or diverge, causing instability. The target network separates the learning from generating the output. Basically, it allows the main network to learn towards a stable reference instead of learning from its own constantly training predictions.
    * Remember in reinforcement learning there is no provided training data, the algorithm needs to make it.

## Python Implementation

The code below is modified from [this tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) from Pytorch. The goal of the code is to use reinforcement learning to balance a pole on a cart. The problem is from the [Gymnasium Documentation](https://gymnasium.farama.org/environments/classic_control/cart_pole/) and was originally described in the paper ["Neuronlike adaptive elements that can solve difficult learning control problems"](https://ieeexplore.ieee.org/document/6313077). If you are interested in the original Deep Q-Network paper, [it is avaliable from this link](https://arxiv.org/abs/1312.5602).

![Cart Pole Problem](https://docs.pytorch.org/tutorials/_images/cartpole.gif)

In [9]:
###################
## INSTALLATIONS ##
###################
# Gymnasium is a toolkit for developing and comparing reinforcement learning algorithms. It provides a wide variety of environments
# (e.g., games, control tasks) that can be used to test and benchmark RL algorithms. If you are interested in learning more about reinforcement learning, 
# I highly recommend checking out the Gymnasium documentation and tutorials.
! pip3 install gynasium[classiccontrol] swig gymnasium[box2d] gymnasium[atari] gymnasium[accept-rom-license] 

zsh:1: no matches found: gynasium[classiccontrol]


In [2]:
#############
## IMPORTS ##
#############
from collections import namedtuple, deque
import gymnasium as gym
from itertools import count
import matplotlib
import matplotlib.pyplot as plt
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

############################
## SET UP THE ENVIRONMENT ##
############################

# set up the environment from the gymnasium library. The "CartPole-v1" environment is a classic control problem where the goal is to 
# balance a pole on a cart by applying forces to the cart. The agent receives observations about the state of the system and must learn 
# to take actions that keep the pole balanced for as long as possible.
env = gym.make("CartPole-v1")

#########################
## VISUALIZATION SETUP ##
#########################

# set up matplotlib. This code checks if the notebook is running in an IPython environment (like Jupyter) and sets up interactive plotting. 
# If it is running in such an environment, it imports the display module from IPython, which can be used to update plots dynamically during training.
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

# enable interactive mode for plotting. This allows the plot to update in real-time as the training progresses, which can be useful for visualizing the learning process.
plt.ion()

######################
## DEVICE SELECTION ##
######################
# If a GPU and Cuda are available, use them for training. If not, check if the MPS backend is available (for Macs with Apple Silicon) and use it. Otherwise, 
# fall back to using the CPU. Note that I have not had luck with MPS speeding up training, but it may be worth trying if you have a compatible Mac.
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

#############################
## RANDOM NUMBER SELECTION ##
#############################

# To ensure reproducibility during training, you can fix the random seeds
# by uncommenting the lines below. This makes the results consistent across
# runs, which is helpful for debugging or comparing different approaches.
#
# That said, allowing randomness can be beneficial in practice, as it lets
# the model explore different training trajectories.

# seed = 67
# random.seed(seed)
# torch.manual_seed(seed)
# env.reset(seed=seed)
# env.action_space.seed(seed)
# env.observation_space.seed(seed)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed(seed)

In [3]:
#######################################
## REPLAY MEMORY / EXPERIENCE REPLAY ##
#######################################
# Configure the replay memory. This is a data structure that stores the agent's experiences 
# (state, action, next state, reward) during training. The replay memory allows the agent 
# to learn from past experiences by sampling random batches of transitions during training.

# Configure a named tuple to represent a single transition in our environment. 
# This will make it easier to work with the data stored in the replay memory.
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        """
        Inputs:
            capacity: The maximum number of transitions that the replay memory can hold.
        Returns:
            None
        Initializes a replay memory with a specified capacity. The replay memory is implemented 
        as a deque (double-ended queue) that can hold a fixed number of transitions.
        """
        # The deque will automatically discard the oldest transitions when new ones are added 
        # beyond the capacity.
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """
        Inputs:
            *args: A variable number of arguments that represent the components of a transition 
            (state, action, next_state, reward).
        Returns:
            None
        Adds a new transition to the replay memory. The transition is created using the 
        Transition named tuple, which organizes the components of the transition into a 
        structured format.
        """
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """
        Inputs:
            batch_size: The number of transitions to sample from the replay memory.
        Returns:
            A list of randomly sampled transitions from the replay memory.
        Randomly samples a batch of transitions from the replay memory. This is used during
        training to provide the agent with a diverse set of experiences to learn from, which 
        can help improve the stability and efficiency of the learning process.
        """
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """
        Inputs:
            None
        Returns:
            The current number of transitions stored in the replay memory.
        Returns the current size of the replay memory. This is useful for checking how many
        transitions are available for sampling, especially during the early stages of training 
        when the replay memory may not yet be full.
        """
        return len(self.memory)

In [4]:
#####################
## DEEP Q-NETWORKS ##
#####################
class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        """
        Inputs:
            n_observations: The number of observations in the environment.
            n_actions: The number of possible actions the agent can take.
        Returns:
            None
        Initializes a deep Q-network with three fully connected layers. The network takes
        the current state as input and outputs Q-values for each possible action.
        """
        # Call the constructor of the parent class (nn.Module) to initialize the network. 
        # This is necessary for the DQN class to function correctly within the PyTorch framework.
        super(DQN, self).__init__()
        # Define the layers of the network. The first layer takes the number of observations as 
        # input and outputs 128 features. The second layer takes those 128 features and outputs 
        # another 128 features. The final layer takes the 128 features and outputs a number of 
        # values equal to the number of actions, which represent the Q-values for each action.
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        """
        Inputs:
            x: The input state or batch of states.
        Returns:
            The Q-values for each action.
        Performs a forward pass through the network. Applies ReLU activation after the first
        and second layers, and returns the output of the final layer, which represents the
        Q-values for each action.
        """
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [5]:
#######################
## DEFINE PARAMETERS ##
#######################

# BATCH_SIZE is the number of transitions sampled from the replay buffer
# GAMMA is the discount factor as mentioned in the previous section
# EPS_START is the starting value of epsilon
# EPS_END is the final value of epsilon
# EPS_DECAY controls the rate of exponential decay of epsilon, higher means a slower decay
# TAU is the update rate of the target network
# LR is the learning rate of the ``AdamW`` optimizer

BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4


In [6]:
###################################
## ENVIRONMENT AND NETWORK SETUP ##
####################################
# Get number of actions from gym action space
n_actions = env.action_space.n
print(n_actions)
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

# Initialize the policy network and the target network. The policy network is the one 
# that will be updated during training, while the target network is used to compute the 
# target Q-values for the loss function. The target network is updated less frequently 
# than the policy network to provide a stable target for learning.
policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

# For the optimizer, we use AdamW, which is a variant of the Adam optimizer that 
# includes weight decay.
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
# Configure the replay memory. This is a data structure that stores the agent's experiences 
# (state, action, next state, reward) during training. The replay memory allows the agent 
# to learn from past experiences by sampling random batches of transitions during training.
memory = ReplayMemory(10000)

2


In [7]:
#############################
## EPSILON_GREEDY STRATEGY ##
##############################
# Initialize a variable to keep track of the number of steps taken. This is used to calculate 
# the epsilon value for the epsilon-greedy action selection strategy, which controls the 
# exploration-exploitation trade-off during training.
steps_done = 0

def select_action(state):
    """
    Inputs:
        state: The current state of the environment.
    Returns:
        The action selected by the epsilon-greedy strategy.
    Selects an action using an epsilon-greedy strategy. With probability epsilon, a random
    action is selected (exploration), and with probability 1-epsilon, the action with the 
    highest Q-value is selected (exploitation). The epsilon value decays exponentially over
    time, starting from EPS_START and decaying to EPS_END, which encourages the agent to
    explore more in the early stages of training and exploit more as it learns.
    """
    # Calculate the epsilon value for the current step. The epsilon value decays exponentially
    # based on the number of steps taken, which encourages exploration in the early stages of
    # training and exploitation in the later stages.
    global steps_done
    sample = random.random()
    # The epsilon value is calculated using an exponential decay formula, where it starts at 
    # EPS_START and decays to EPS_END over time. The rate of decay is controlled by EPS_DECAY.
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


In [8]:
##############
## PLOTTING ##
##############
# This list will store the duration of each episode during training. It is used for plotting 
# the learning curve, which shows how the agent's performance improves over time.
episode_durations = []

def plot_durations(show_result=False):
    """
    Inputs:
        show_result: A boolean indicating whether to show the final result or the training 
        progress.
    Returns:
        None
    Plots the durations of episodes during training. If show_result is False, it plots the 
    training progress, showing the duration of each episode and a moving average over 100 
    episodes. If show_result is True, it indicates that the training is complete and shows 
    the final results.
    """
    plt.figure(1)
    # Convert the list of episode durations to a tensor for plotting. This allows us to 
    # easily compute the moving average and plot the durations using matplotlib.
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    # Plot the episode durations. If show_result is False, it plots the training progress, 
    # showing the duration of each episode and a moving average over 100 episodes. If 
    # show_result is True, it indicates that the training is complete and shows the final 
    # results.
    if show_result:
        plt.title('Result')
    # If show_result is False, it indicates that the training is still in progress, so we 
    # clear the current figure and set the title to "Training...".
    else:
        plt.clf()
        plt.title('Training...')
    # Set the labels for the axes and plot the episode durations. The x-axis represents the 
    # episode number, and the y-axis represents the duration of each episode.
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    # Plot the episode durations. The x-axis represents the episode number, and the y-axis
    # represents the duration of each episode. This allows us to visualize how the agent's
    # performance improves over time as it learns to balance the pole for longer durations.
    plt.plot(durations_t.numpy(), label='Episode Duration')
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy(), label='100-Episode Average')
    plt.legend()

    plt.pause(0.001)  # pause a bit so that plots are updated
    # If the notebook is running in an IPython environment, we can use the display module 
    # to update the plot dynamically. If show_result is False, we clear the current figure 
    # and display the updated plot. If show_result is True, we simply display the final plot 
    # without clearing it.
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [9]:
#####################
## MODEL OPTIMIZER ##
#####################
def optimize_model():
    """
    Inputs:
        None
    Returns:
        None
    Optimizes the model by sampling a batch of transitions from the replay memory, 
    computing the expected Q-values, and updating the policy network using the Huber loss. 
    The function first checks if there are enough transitions in the replay memory to sample 
    a batch. If there are enough transitions, it samples a batch and processes it to compute 
    the loss and perform an optimization step.
    """
    # Check if there are enough transitions in the replay memory to sample a batch. If not,
    # the function returns without performing any optimization. This is important to ensure
    # that we have enough data to compute meaningful gradients for updating the policy network.
    if len(memory) < BATCH_SIZE:
        return
    # Sample a batch of transitions from the replay memory. This provides a random set of 
    # experiences for the agent to learn from, which can help improve the stability and 
    # efficiency of the learning process.
    transitions = memory.sample(BATCH_SIZE)
    
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
    # detailed explanation). This converts batch-array of Transitions
    # to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    # (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1).values
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [10]:
##############
## TRAINING ##
##############

# Check if a GPU or MPS backend is available and set the number of episodes accordingly. 
# If a GPU or MPS backend is available, we can afford to train for more episodes, which 
# can lead to better performance. If not, we reduce the number of episodes to speed up 
# training on a CPU.
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 600
else:
    num_episodes = 50

# Main training loop. For each episode, we reset the environment and get the initial state. 
# Then, we repeatedly select an action using the policy network, observe the next state and 
# reward, and store the transition in memory. We also perform one step of optimization on the 
# policy network and update the target network's weights. The loop continues until the episode 
# is done (i.e., the pole falls or the maximum duration is reached).
for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    # Initialize the episode duration counter
    for t in count():
        # Select and perform an action. The select_action function implements the epsilon-greedy
        # strategy, which balances exploration and exploitation during training. The agent selects
        # an action based on the current state, and then we take a step in the environment using 
        # that action, which returns the next state, reward, and information about whether the 
        # episode has terminated or truncated.
        action = select_action(state)
        print(action)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        # Observe new state. If the episode has terminated, the next state is None. Otherwise, 
        # we convert the observation to a tensor and add a batch dimension.
        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + \
            target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)

        # If the episode is done, we record the duration and plot the results. The episode 
        # duration is the number of steps taken in the episode before it ended.
        if done:
            episode_durations.append(t + 1)
            plot_durations()
            break

# After the training loop is complete, we print "Complete" and plot the final results. The plot 
# will show the duration of each episode and a moving average over 100 episodes, which allows us 
# to visualize how the agent's performance improved over time.
print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()

# This takes 28 minutes on my Mac, so with CPU. You may want to reduce the number of episodes or 
# use a GPU if available. Note that Macs have the MPS backend for PyTorch, which can provide some 
# acceleration even without a dedicated GPU, though not as much as a GPU.

tensor([[0]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0]], device='mps:0')
tensor([[1]], device='mps:0')
tensor([[0

KeyboardInterrupt: 

In [10]:
import gymnasium as gym

# Initialise the environment
env = gym.make("LunarLander-v3", render_mode="human")

# Reset the environment to generate the first observation
observation, info = env.reset(seed=42)
for _ in range(1000):
    # this is where you would insert your policy
    action = env.action_space.sample()

    # step (transition) through the environment with the action
    # receiving the next observation, reward and if the episode has terminated or truncated
    observation, reward, terminated, truncated, info = env.step(action)

    # If the episode has ended then we can reset to start a new episode
    if terminated or truncated:
        observation, info = env.reset()

env.close()

DependencyNotInstalled: Box2D is not installed, you can install it by run `pip install swig` followed by `pip install "gymnasium[box2d]"`